# DEE — Defectos BLANDOS (Test 2B revisado)

**Contexto:** el test con defectos duros (×5 y ×10) mostró localización inversa: los modos se concentran FUERA del defecto porque la rigidez aumentada actúa como barrera repulsiva para los modos de baja frecuencia.

**Hipótesis:** defectos BLANDOS (factor <1) deberían actuar como pozos atractivos, capturando modos de baja frecuencia. Debería aparecer al menos un modo con ω **por debajo** del primer sexteto del cristal perfecto y localizado DENTRO del defecto.

**Tres configuraciones (mismas geometrías, factores invertidos):**
1. Dislocación 1D: factor ×0.2
2. Plano 2D: factor ×0.2
3. Inclusión 3D: factor ×0.1 (más extremo para superar umbral Lifshitz)

---

In [ ]:
import os
import numpy as np
from scipy.sparse import csr_matrix, diags
from scipy.sparse.linalg import eigsh
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import time

PLOT_DIR = 'plots_defectos_blandos'
os.makedirs(PLOT_DIR, exist_ok=True)

def save_plot(name, fig=None, dpi=120):
    if fig is None: fig = plt.gcf()
    path = os.path.join(PLOT_DIR, f'{name}.png')
    fig.savefig(path, dpi=dpi, bbox_inches='tight')
    print(f'  → {path}')

print('Setup listo')

In [ ]:
def periodic_distance_matrix(points, L=1.0):
    diff = points[:, None, :] - points[None, :, :]
    diff = diff - L * np.round(diff / L)
    return np.linalg.norm(diff, axis=-1)

def grid_perturbed_T3(N_target, jitter_fraction, seed=42):
    rng = np.random.default_rng(seed)
    n_side = round(N_target**(1/3))
    N_actual = n_side**3
    spacing = 1.0 / n_side
    coords = np.arange(n_side) * spacing + spacing/2
    gx, gy, gz = np.meshgrid(coords, coords, coords, indexing='ij')
    points = np.stack([gx.ravel(), gy.ravel(), gz.ravel()], axis=1)
    points += rng.uniform(-jitter_fraction*spacing, jitter_fraction*spacing, size=points.shape)
    points = points % 1.0
    return points, N_actual

def build_laplacian_with_force_modifiers(points, k_neighbors=30, alpha=2.0, L=1.0,
                                           force_modifiers=None):
    D_mat = periodic_distance_matrix(points, L=L)
    N = len(points)
    if force_modifiers is None:
        force_modifiers = {}
    D_search = D_mat.copy()
    np.fill_diagonal(D_search, np.inf)
    neighbor_idx = np.argpartition(D_search, k_neighbors, axis=1)[:, :k_neighbors]
    rows, cols, vals = [], [], []
    for i in range(N):
        for j in neighbor_idx[i]:
            d = D_mat[i, j]
            if d > 0:
                w = 1.0 / d**alpha
                mod_i = force_modifiers.get(i, 1.0)
                mod_j = force_modifiers.get(j, 1.0)
                w = w * np.sqrt(mod_i * mod_j)
                rows.append(i); cols.append(j); vals.append(w)
    W = csr_matrix((vals, (rows, cols)), shape=(N, N))
    W = (W + W.T) / 2
    degrees = np.array(W.sum(axis=1)).flatten()
    return diags(degrees) - W

def participation_ratios(eigenvectors):
    parts = []
    for i in range(eigenvectors.shape[1]):
        v = eigenvectors[:, i] / np.linalg.norm(eigenvectors[:, i])
        v2 = v**2
        ipr = np.sum(v2**2)
        parts.append(1.0 / ipr if ipr > 0 else len(v))
    return np.array(parts)

# Construcción
N_TARGET = 1331
K_NEIGHBORS = 30
JITTER = 0.1

points, N = grid_perturbed_T3(N_TARGET, JITTER, seed=42)
L_perfect = build_laplacian_with_force_modifiers(points, K_NEIGHBORS, alpha=2.0)

t0 = time.time()
eigs_perfect, vecs_perfect = eigsh(L_perfect, k=60, which='SM', tol=1e-8)
idx_s = np.argsort(eigs_perfect)
eigs_perfect = eigs_perfect[idx_s]
vecs_perfect = vecs_perfect[:, idx_s]
omegas_perfect = np.sqrt(eigs_perfect[eigs_perfect > 1e-3])
parts_perfect = participation_ratios(vecs_perfect)
print(f'Cristal perfecto: {time.time()-t0:.1f}s')
print(f'ω mínima del cristal perfecto (inicio del primer sexteto): {omegas_perfect[0]:.3f}')
print(f'Primeros 10 ω: {omegas_perfect[:10].round(3)}')

## Tres defectos blandos

In [ ]:
# 1. Dislocación blanda
y_center, z_center = 0.5, 0.5
dy = points[:, 1] - y_center; dy -= np.round(dy)
dz = points[:, 2] - z_center; dz -= np.round(dz)
dist_to_axis = np.sqrt(dy**2 + dz**2)
dislocation_nodes = np.where(dist_to_axis < 0.08)[0]

DISLOC_SOFT = 0.2
mods_disloc = {int(i): DISLOC_SOFT for i in dislocation_nodes}

t0 = time.time()
L_disloc = build_laplacian_with_force_modifiers(points, K_NEIGHBORS, 2.0, force_modifiers=mods_disloc)
eigs_disloc, vecs_disloc = eigsh(L_disloc, k=60, which='SM', tol=1e-8)
idx = np.argsort(eigs_disloc)
eigs_disloc = eigs_disloc[idx]; vecs_disloc = vecs_disloc[:, idx]
omegas_disloc = np.sqrt(eigs_disloc[eigs_disloc > 1e-3])
parts_disloc = participation_ratios(vecs_disloc)
print(f'Dislocación blanda (×{DISLOC_SOFT}): {time.time()-t0:.1f}s, {len(dislocation_nodes)} nodos modificados')

# 2. Plano blando
dist_to_plane = np.abs(points[:, 2] - 0.5)
plane_nodes = np.where(dist_to_plane < 0.03)[0]

PLANE_SOFT = 0.2
mods_plane = {int(i): PLANE_SOFT for i in plane_nodes}

t0 = time.time()
L_plane = build_laplacian_with_force_modifiers(points, K_NEIGHBORS, 2.0, force_modifiers=mods_plane)
eigs_plane, vecs_plane = eigsh(L_plane, k=60, which='SM', tol=1e-8)
idx = np.argsort(eigs_plane)
eigs_plane = eigs_plane[idx]; vecs_plane = vecs_plane[:, idx]
omegas_plane = np.sqrt(eigs_plane[eigs_plane > 1e-3])
parts_plane = participation_ratios(vecs_plane)
print(f'Plano blando (×{PLANE_SOFT}): {time.time()-t0:.1f}s, {len(plane_nodes)} nodos modificados')

# 3. Inclusión blanda (más extrema para 3D)
center = np.array([0.5, 0.5, 0.5])
diffs = points - center; diffs -= np.round(diffs)
dist_center = np.linalg.norm(diffs, axis=1)
inclusion_nodes = np.where(dist_center < 0.15)[0]

INCL_SOFT = 0.1
mods_incl = {int(i): INCL_SOFT for i in inclusion_nodes}

t0 = time.time()
L_incl = build_laplacian_with_force_modifiers(points, K_NEIGHBORS, 2.0, force_modifiers=mods_incl)
eigs_incl, vecs_incl = eigsh(L_incl, k=80, which='SM', tol=1e-8)
idx = np.argsort(eigs_incl)
eigs_incl = eigs_incl[idx]; vecs_incl = vecs_incl[:, idx]
omegas_incl = np.sqrt(eigs_incl[eigs_incl > 1e-3])
parts_incl = participation_ratios(vecs_incl)
print(f'Inclusión blanda (×{INCL_SOFT}): {time.time()-t0:.1f}s, {len(inclusion_nodes)} nodos modificados')

In [ ]:
# Primera señal clave: ¿aparecen frecuencias por DEBAJO del cristal perfecto?
omega_min_perfect = omegas_perfect[0]

print('='*60)
print(f'ω mínima del cristal perfecto: {omega_min_perfect:.3f}')
print('='*60)
print()

for name, omegas, parts, n_defect in [('Dislocación', omegas_disloc, parts_disloc, len(dislocation_nodes)),
                                       ('Plano', omegas_plane, parts_plane, len(plane_nodes)),
                                       ('Inclusión', omegas_incl, parts_incl, len(inclusion_nodes))]:
    n_below = np.sum(omegas < omega_min_perfect * 0.98)
    print(f'{name} (×{DISLOC_SOFT if name=="Dislocación" else (PLANE_SOFT if name=="Plano" else INCL_SOFT)}):')
    print(f'  Nodos en el defecto: {n_defect}')
    print(f'  ω mínima: {omegas[0]:.3f}')
    print(f'  Modos con ω < {omega_min_perfect:.3f} (por debajo del cristal): {n_below}')
    if n_below > 0:
        print(f'  ω de esos modos: {omegas[:n_below].round(3)}')
        print(f'  Participation ratios: {parts[:n_below].round(1)}')
    print()

In [ ]:
# Plots: espectros comparados
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

ax = axes[0, 0]
n_plot = 30
ax.plot(np.arange(n_plot), omegas_perfect[:n_plot], 'o', color='steelblue',
        markersize=8, alpha=0.7, label='Cristal perfecto')
ax.plot(np.arange(n_plot), omegas_disloc[:n_plot], '^', color='darkgreen',
        markersize=8, alpha=0.7, label=f'Dislocación blanda (×{DISLOC_SOFT})')
ax.plot(np.arange(n_plot), omegas_plane[:n_plot], 's', color='orange',
        markersize=8, alpha=0.7, label=f'Plano blando (×{PLANE_SOFT})')
ax.plot(np.arange(n_plot), omegas_incl[:n_plot], 'v', color='crimson',
        markersize=8, alpha=0.7, label=f'Inclusión blanda (×{INCL_SOFT})')
ax.axhline(omega_min_perfect, color='blue', linestyle=':', alpha=0.5,
           label='ω mínima del cristal perfecto')
ax.set_xlabel('Índice del modo')
ax.set_ylabel('ω')
ax.set_title('Espectros — ¿hay modos bajo el cristal perfecto?')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Zoom a bajas frecuencias
ax = axes[0, 1]
ax.plot(np.arange(15), omegas_perfect[:15], 'o-', color='steelblue', markersize=10, alpha=0.7,
        label='Cristal perfecto')
ax.plot(np.arange(15), omegas_disloc[:15], '^-', color='darkgreen', markersize=10, alpha=0.7,
        label='Dislocación')
ax.plot(np.arange(15), omegas_plane[:15], 's-', color='orange', markersize=10, alpha=0.7,
        label='Plano')
ax.plot(np.arange(15), omegas_incl[:15], 'v-', color='crimson', markersize=10, alpha=0.7,
        label='Inclusión')
ax.axhline(omega_min_perfect, color='blue', linestyle=':', alpha=0.5)
ax.set_xlabel('Índice del modo (zoom bajas frecuencias)')
ax.set_ylabel('ω')
ax.set_title('Zoom: primeros 15 modos')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Participation ratio vs ω
ax = axes[1, 0]
ax.semilogy(omegas_perfect[:n_plot], parts_perfect[:n_plot], 'o', color='steelblue',
            markersize=8, alpha=0.6, label='Perfecto')
ax.semilogy(omegas_disloc[:n_plot], parts_disloc[:n_plot], '^', color='darkgreen',
            markersize=8, alpha=0.7, label='Dislocación')
ax.semilogy(omegas_plane[:n_plot], parts_plane[:n_plot], 's', color='orange',
            markersize=8, alpha=0.7, label='Plano')
ax.semilogy(omegas_incl[:n_plot], parts_incl[:n_plot], 'v', color='crimson',
            markersize=8, alpha=0.7, label='Inclusión')
ax.axhline(N/10, color='red', linestyle='--', label=f'Umbral N/10 = {int(N/10)}')
ax.set_xlabel('ω')
ax.set_ylabel('Participation ratio')
ax.set_title('Participation ratio vs ω')
ax.legend(fontsize=9)
ax.grid(alpha=0.3, which='both')

# Comparación con defectos duros anteriores
ax = axes[1, 1]
min_parts = [parts_perfect[:50].min(),
             parts_disloc[:50].min(),
             parts_plane[:50].min(),
             parts_incl[:50].min()]
labels = ['Perfecto', 'Dislocación', 'Plano', 'Inclusión']
colors_bar = ['steelblue', 'darkgreen', 'orange', 'crimson']
bars = ax.bar(labels, min_parts, color=colors_bar, alpha=0.7)
ax.axhline(N/10, color='red', linestyle='--', label=f'Umbral localización N/10 = {int(N/10)}')
ax.set_ylabel('Min participation ratio (primeros 50)')
ax.set_title('Localización: ¿alguno cruza el umbral?')
ax.legend()
ax.grid(alpha=0.3, axis='y')
for bar, val in zip(bars, min_parts):
    ax.text(bar.get_x() + bar.get_width()/2, val + 20, f'{val:.0f}',
            ha='center', fontsize=10)

plt.tight_layout()
save_plot('26_defectos_blandos_comparacion')
plt.show()

In [ ]:
# Visualización 3D del modo de MÁS BAJA frecuencia para cada defecto
fig = plt.figure(figsize=(18, 5))

configs = [
    ('Dislocación ×0.2', vecs_disloc, parts_disloc, omegas_disloc, dislocation_nodes),
    ('Plano ×0.2', vecs_plane, parts_plane, omegas_plane, plane_nodes),
    ('Inclusión ×0.1', vecs_incl, parts_incl, omegas_incl, inclusion_nodes)
]

for plot_idx, (name, vecs, parts, omegas, defect_nodes) in enumerate(configs):
    # El modo de más baja frecuencia (excluyendo modo cero)
    mode_idx = 0 if vecs.shape[1] > 0 else -1
    # Saltar modo cero si el primer autovalor es casi cero
    eigs_this = eigs_disloc if plot_idx == 0 else (eigs_plane if plot_idx == 1 else eigs_incl)
    if eigs_this[0] < 1e-3:
        mode_idx = 1
    
    v = vecs[:, mode_idx]
    v2 = v**2
    log_v2 = np.log10(v2 + 1e-12)
    
    ax = fig.add_subplot(1, 3, plot_idx+1, projection='3d')
    sc = ax.scatter(points[:, 0], points[:, 1], points[:, 2],
                     c=log_v2, s=15, alpha=0.4, cmap='hot')
    ax.scatter(points[defect_nodes, 0], points[defect_nodes, 1],
               points[defect_nodes, 2], c='cyan', s=30, alpha=0.8,
               edgecolors='black', linewidths=0.5)
    ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
    # Extraer ω y part correctos
    om_this = np.sqrt(eigs_this[mode_idx])
    part_this = parts[mode_idx]
    ax.set_title(f'{name}\nModo más bajo: ω={om_this:.3f}, part={part_this:.1f}',
                 fontsize=11)
    plt.colorbar(sc, ax=ax, shrink=0.6, label='log|v|²')

plt.tight_layout()
save_plot('27_modos_mas_bajos_3d')
plt.show()

In [ ]:
# Perfiles radiales del modo más bajo (para cada defecto)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for plot_idx, (name, vecs, parts, omegas, defect_nodes) in enumerate(configs):
    ax = axes[plot_idx]
    eigs_this = eigs_disloc if plot_idx == 0 else (eigs_plane if plot_idx == 1 else eigs_incl)
    mode_idx = 1 if eigs_this[0] < 1e-3 else 0
    v = vecs[:, mode_idx]
    v2 = v**2
    
    defect_center = points[defect_nodes].mean(axis=0)
    diffs = points - defect_center
    diffs = diffs - np.round(diffs)
    
    # Para dislocación medimos distancia al eje x; para plano al plano z; para esfera al centro
    if name.startswith('Dislocación'):
        dist = np.sqrt(diffs[:, 1]**2 + diffs[:, 2]**2)
        xlabel = 'Distancia al eje x del tubo'
    elif name.startswith('Plano'):
        dist = np.abs(diffs[:, 2])
        xlabel = 'Distancia al plano z=0.5'
    else:
        dist = np.linalg.norm(diffs, axis=1)
        xlabel = 'Distancia al centro de la inclusión'
    
    # Bineado radial
    r_bins = np.linspace(0, 0.5, 20)
    r_centers = 0.5 * (r_bins[:-1] + r_bins[1:])
    v2_by_r = np.zeros(len(r_centers))
    counts = np.zeros(len(r_centers))
    for k, r in enumerate(r_centers):
        mask = (dist >= r_bins[k]) & (dist < r_bins[k+1])
        if mask.sum() > 0:
            v2_by_r[k] = v2[mask].mean()
            counts[k] = mask.sum()
    
    mask_valid = counts > 0
    om_this = np.sqrt(eigs_this[mode_idx])
    color = ['darkgreen', 'orange', 'crimson'][plot_idx]
    ax.semilogy(r_centers[mask_valid], v2_by_r[mask_valid], 'o-',
                color=color, markersize=8, lw=2)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('⟨|v|²⟩')
    ax.set_title(f'{name}: modo más bajo ω={om_this:.3f}')
    ax.grid(alpha=0.3, which='both')

plt.tight_layout()
save_plot('28_perfiles_radiales_blandos')
plt.show()

In [ ]:
# Veredicto final
print('='*70)
print('VEREDICTO — Defectos blandos')
print('='*70)
print()
print('Criterios:')
print('  (A) ¿Aparecen frecuencias bajo ω mínima del cristal perfecto?')
print('  (B) ¿Los modos bajos decaen en amplitud lejos del defecto?')
print('  (C) ¿Participation ratio de esos modos es bajo?')
print()

omega_min_ref = omegas_perfect[0]

for name, omegas, parts, n_defect in [('Dislocación', omegas_disloc, parts_disloc, len(dislocation_nodes)),
                                        ('Plano', omegas_plane, parts_plane, len(plane_nodes)),
                                        ('Inclusión', omegas_incl, parts_incl, len(inclusion_nodes))]:
    n_sub = np.sum(omegas < omega_min_ref * 0.98)
    min_om = omegas[0]
    min_part = parts[:10].min()
    criterio_A = n_sub > 0
    criterio_C_part = min_part < N / 10
    criterio_C_moderate = min_part < N / 4
    
    print(f'{name}:')
    print(f'  ω mínima: {min_om:.3f} (vs {omega_min_ref:.3f} perfecto)')
    print(f'  Modos bajo cristal perfecto: {n_sub}')
    print(f'  Min participation: {min_part:.0f} (umbral fuerte N/10={int(N/10)}, moderado N/4={int(N/4)})')
    if criterio_A and criterio_C_part:
        print(f'  → ✓✓ LOCALIZACIÓN FUERTE confirmada')
    elif criterio_A and criterio_C_moderate:
        print(f'  → ✓ localización moderada detectada')
    elif criterio_A:
        print(f'  → ~ modos bajos presentes pero no localizados claramente')
    else:
        print(f'  → ✗ sin evidencia clara de localización')
    print()

In [ ]:
import shutil
shutil.make_archive('plots_defectos_blandos', 'zip', PLOT_DIR)
print(f'\nZip creado: plots_defectos_blandos.zip')

try:
    from google.colab import files
    files.download('plots_defectos_blandos.zip')
    print('→ Descarga iniciada')
except ImportError:
    pass